# Main Table Results - Bounding Box Evaluation

This notebook generates the main results table from the bounding box evaluation results in `/shared_data0/weiqiuy/llm_cholec_organ/results/bbox_cholecseg8k_local_quick/`.

In [1]:
import itertools
import pandas as pd
import json
from pathlib import Path
import numpy as np
from typing import List, Tuple

In [2]:
# -----------------------------
# 1) Configure methods & tasks
# -----------------------------

# Models based on what's actually in the results directory
GENERAL_MODELS = [
    "GPT-4.1",
    "Gemini-2.0-Flash", 
    "Claude-Sonnet-4",
    "Llava-v1.6-Mistral-7B",
    "Qwen2.5-VL-7B",
    "Pixtral-12B"
]

# Mapping from directory names to display names
MODEL_NAME_MAPPING = {
    "gpt-4.1": "GPT-4.1",
    "gemini-2.0-flash": "Gemini-2.0-Flash",
    "claude-sonnet-4-20250514": "Claude-Sonnet-4",
    "llava-hf_llava-v1.6-mistral-7b-hf": "Llava-v1.6-Mistral-7B",
    "Qwen_Qwen2.5-VL-7B-Instruct": "Qwen2.5-VL-7B",
    "mistralai_Pixtral-12B-2409": "Pixtral-12B"
}

# For this initial version, we'll focus on CholecSeg8k results
TASKS = {
    "CholecSeg8k": ["Presence", "IoU"]
}

ALL_METHODS = GENERAL_MODELS

In [3]:
# -----------------------------
# 2) Load results from JSON files with Bootstrap
# -----------------------------

results_dir = Path("/shared_data0/weiqiuy/llm_cholec_organ/results/bbox_cholecseg8k_local_quick")

def bootstrap_std(data: List[float], n_bootstrap: int = 1000, seed: int = 42) -> float:
    """Compute bootstrap standard deviation of the mean."""
    np.random.seed(seed)
    data = np.array(data)
    n = len(data)
    
    if n == 0:
        return 0.0
    
    bootstrap_means = []
    for _ in range(n_bootstrap):
        # Sample with replacement
        sample = np.random.choice(data, size=n, replace=True)
        bootstrap_means.append(np.mean(sample))
    
    return np.std(bootstrap_means)

def load_summary_results(mode="zeroshot_combined", compute_bootstrap=True):
    """Load results from individual test files and compute metrics with bootstrap confidence intervals.
    Now loads BOTH bbox-to-bbox and bbox-to-mask IoU values.
    
    Args:
        mode: Evaluation mode (e.g., "zeroshot_combined")
        compute_bootstrap: If True, compute bootstrap standard deviations
    """
    mode_dir = results_dir / mode
    results = {}
    
    if not mode_dir.exists():
        print(f"Warning: {mode_dir} does not exist")
        return results
    
    for model_dir in mode_dir.iterdir():
        if not model_dir.is_dir():
            continue
            
        model_name = model_dir.name
        display_name = MODEL_NAME_MAPPING.get(model_name, model_name)
        
        # Load all test result files
        test_files = list(model_dir.glob("test_*.json"))
        
        if not test_files:
            print(f"No test files found for {model_name}")
            continue
        
        all_y_true = []
        all_y_pred = []
        all_ious_bbox = []  # bbox-to-bbox IoU
        all_ious_mask = []  # bbox-to-mask IoU
        all_binary_correct = []  # For bootstrap of presence accuracy
        
        for test_file in test_files:
            with open(test_file, 'r') as f:
                data = json.load(f)
            
            # Collect presence accuracy data
            if 'y_true' in data and 'y_pred' in data:
                all_y_true.extend(data['y_true'])
                all_y_pred.extend(data['y_pred'])
                # Store binary correctness for each prediction
                all_binary_correct.extend([1 if yt == yp else 0 
                                          for yt, yp in zip(data['y_true'], data['y_pred'])])
            
            # Collect BOTH IoU types from individual organ data
            for organ in data.get('organs', []):
                # Bbox-to-mask IoU
                if 'iou_bbox_to_mask' in organ and organ['iou_bbox_to_mask'] is not None:
                    all_ious_mask.append(organ['iou_bbox_to_mask'])
                
                # Bbox-to-bbox IoU (stored as 'iou' in the organ data)
                if 'iou' in organ and organ['iou'] is not None:
                    all_ious_bbox.append(organ['iou'])
        
        # Compute metrics
        metrics = {}
        
        # Presence accuracy with bootstrap
        if all_binary_correct:
            metrics['presence_accuracy'] = np.mean(all_binary_correct)
            if compute_bootstrap:
                metrics['presence_accuracy_std'] = bootstrap_std(all_binary_correct)
        
        # Bbox-to-bbox IoU with bootstrap
        if all_ious_bbox:
            metrics['mean_iou_bbox'] = np.mean(all_ious_bbox)
            metrics['iou_at_50_bbox'] = np.mean([iou >= 0.5 for iou in all_ious_bbox])
            
            if compute_bootstrap:
                metrics['mean_iou_bbox_std'] = bootstrap_std(all_ious_bbox)
                iou_at_50_binary = [1 if iou >= 0.5 else 0 for iou in all_ious_bbox]
                metrics['iou_at_50_bbox_std'] = bootstrap_std(iou_at_50_binary)
        
        # Bbox-to-mask IoU with bootstrap
        if all_ious_mask:
            metrics['mean_iou_mask'] = np.mean(all_ious_mask)
            metrics['iou_at_50_mask'] = np.mean([iou >= 0.5 for iou in all_ious_mask])
            
            if compute_bootstrap:
                metrics['mean_iou_mask_std'] = bootstrap_std(all_ious_mask)
                iou_at_50_binary = [1 if iou >= 0.5 else 0 for iou in all_ious_mask]
                metrics['iou_at_50_mask_std'] = bootstrap_std(iou_at_50_binary)
        
        results[display_name] = metrics
        print(f"Loaded {len(test_files)} files for {display_name} (both IoU types)")
    
    return results

# Load zeroshot_combined results with both IoU types and bootstrap
print("Loading results with BOTH bbox-to-bbox and bbox-to-mask IoU with Bootstrap...")
zeroshot_results = load_summary_results("zeroshot_combined", compute_bootstrap=True)
print(f"\nLoaded results for {len(zeroshot_results)} models with bootstrap std")

Loading results with BOTH bbox-to-bbox and bbox-to-mask IoU with Bootstrap...
Loaded 200 files for GPT-4.1 (both IoU types)
Loaded 200 files for Gemini-2.0-Flash (both IoU types)
Loaded 200 files for Llava-v1.6-Mistral-7B (both IoU types)
Loaded 200 files for Claude-Sonnet-4 (both IoU types)
Loaded 200 files for Pixtral-12B (both IoU types)
Loaded 200 files for Qwen2.5-VL-7B (both IoU types)

Loaded results for 6 models with bootstrap std


In [4]:
# -----------------------------
# 3) Build the results table with both IoU types (with best/second formatting)
# -----------------------------

# MultiIndex columns: (Task, Metric)
columns = pd.MultiIndex.from_tuples(
    [("CholecSeg8k", "Presence"), 
     ("CholecSeg8k", "IoU (BBox)"), 
     ("CholecSeg8k", "IoU@0.5 (BBox)"),
     ("CholecSeg8k", "IoU (Mask)"), 
     ("CholecSeg8k", "IoU@0.5 (Mask)")],
    names=["Task", "Metric"]
)

# Initialize dataframes
df = pd.DataFrame("", index=ALL_METHODS, columns=columns)
df_raw = pd.DataFrame(np.nan, index=ALL_METHODS, columns=columns)  # For numeric comparisons

# Track maximum standard deviations for summary
max_stds_simple = {
    'presence': 0.0,
    'iou_bbox': 0.0,
    'iou_at_50_bbox': 0.0,
    'iou_mask': 0.0,
    'iou_at_50_mask': 0.0
}

# Fill in the results with only mean values
for model_name in ALL_METHODS:
    if model_name in zeroshot_results:
        metrics = zeroshot_results[model_name]
        
        # Presence accuracy
        if 'presence_accuracy' in metrics:
            mean_val = metrics['presence_accuracy']
            df.loc[model_name, ("CholecSeg8k", "Presence")] = f"{mean_val:.3f}"
            df_raw.loc[model_name, ("CholecSeg8k", "Presence")] = mean_val
            if 'presence_accuracy_std' in metrics:
                max_stds_simple['presence'] = max(max_stds_simple['presence'], metrics['presence_accuracy_std'])
        
        # Bbox-to-bbox IoU
        if 'mean_iou_bbox' in metrics:
            mean_val = metrics['mean_iou_bbox']
            df.loc[model_name, ("CholecSeg8k", "IoU (BBox)")] = f"{mean_val:.3f}"
            df_raw.loc[model_name, ("CholecSeg8k", "IoU (BBox)")] = mean_val
            if 'mean_iou_bbox_std' in metrics:
                max_stds_simple['iou_bbox'] = max(max_stds_simple['iou_bbox'], metrics['mean_iou_bbox_std'])
        
        # Bbox-to-bbox IoU@0.5
        if 'iou_at_50_bbox' in metrics:
            mean_val = metrics['iou_at_50_bbox']
            df.loc[model_name, ("CholecSeg8k", "IoU@0.5 (BBox)")] = f"{mean_val:.3f}"
            df_raw.loc[model_name, ("CholecSeg8k", "IoU@0.5 (BBox)")] = mean_val
            if 'iou_at_50_bbox_std' in metrics:
                max_stds_simple['iou_at_50_bbox'] = max(max_stds_simple['iou_at_50_bbox'], metrics['iou_at_50_bbox_std'])
        
        # Bbox-to-mask IoU
        if 'mean_iou_mask' in metrics:
            mean_val = metrics['mean_iou_mask']
            df.loc[model_name, ("CholecSeg8k", "IoU (Mask)")] = f"{mean_val:.3f}"
            df_raw.loc[model_name, ("CholecSeg8k", "IoU (Mask)")] = mean_val
            if 'mean_iou_mask_std' in metrics:
                max_stds_simple['iou_mask'] = max(max_stds_simple['iou_mask'], metrics['mean_iou_mask_std'])
        
        # Bbox-to-mask IoU@0.5
        if 'iou_at_50_mask' in metrics:
            mean_val = metrics['iou_at_50_mask']
            df.loc[model_name, ("CholecSeg8k", "IoU@0.5 (Mask)")] = f"{mean_val:.3f}"
            df_raw.loc[model_name, ("CholecSeg8k", "IoU@0.5 (Mask)")] = mean_val
            if 'iou_at_50_mask_std' in metrics:
                max_stds_simple['iou_at_50_mask'] = max(max_stds_simple['iou_at_50_mask'], metrics['iou_at_50_mask_std'])
    else:
        # Mark as not available if no results found
        for col in columns:
            df.loc[model_name, col] = "—"

# Find best and second-best for each column
best_second = {}
for col in df_raw.columns:
    col_values = df_raw[col].dropna()
    if len(col_values) >= 2:
        sorted_vals = col_values.nlargest(2)
        best_second[col] = {'best': sorted_vals.iloc[0], 'second': sorted_vals.iloc[1]}
    elif len(col_values) == 1:
        best_second[col] = {'best': col_values.iloc[0], 'second': None}

# Create formatted dataframe
df_formatted = df.copy()
for col in df_raw.columns:
    if col in best_second:
        for idx in df_raw.index:
            if not pd.isna(df_raw.loc[idx, col]):
                value = df_raw.loc[idx, col]
                formatted_value = df.loc[idx, col]
                
                if value == best_second[col]['best']:
                    df_formatted.loc[idx, col] = f"\\textbf{{{formatted_value}}}"
                elif best_second[col]['second'] and value == best_second[col]['second']:
                    df_formatted.loc[idx, col] = f"\\textit{{{formatted_value}}}"

# Display the table
print("\n" + "="*100)
print("MAIN RESULTS TABLE - Zero-shot Combined (BOTH IoU Types)")
print("="*100)
print(df.to_string())
print(f"\nFormatting: Best values in bold, second-best in italics")
print(f"\nMaximum bootstrap standard deviations (1000 samples):")
print(f"  Presence: {max_stds_simple['presence']:.3f}")
print(f"  IoU (BBox): {max_stds_simple['iou_bbox']:.3f}")
print(f"  IoU@0.5 (BBox): {max_stds_simple['iou_at_50_bbox']:.3f}")
print(f"  IoU (Mask): {max_stds_simple['iou_mask']:.3f}")
print(f"  IoU@0.5 (Mask): {max_stds_simple['iou_at_50_mask']:.3f}")


MAIN RESULTS TABLE - Zero-shot Combined (BOTH IoU Types)
Task                  CholecSeg8k                                                    
Metric                   Presence IoU (BBox) IoU@0.5 (BBox) IoU (Mask) IoU@0.5 (Mask)
GPT-4.1                     0.744      0.341          0.256      0.274          0.106
Gemini-2.0-Flash            0.655      0.294          0.192      0.208          0.075
Claude-Sonnet-4             0.688      0.267          0.122      0.179          0.048
Llava-v1.6-Mistral-7B       0.546      0.000          0.000      0.000          0.000
Qwen2.5-VL-7B               0.714      0.367          0.342      0.216          0.118
Pixtral-12B                 0.715      0.090          0.009      0.078          0.000

Formatting: Best values in bold, second-best in italics

Maximum bootstrap standard deviations (1000 samples):
  Presence: 0.010
  IoU (BBox): 0.009
  IoU@0.5 (BBox): 0.015
  IoU (Mask): 0.007
  IoU@0.5 (Mask): 0.011


In [5]:
# -----------------------------
# 4) Generate LaTeX table with formatting (best/second-best)
# -----------------------------

def dataframe_to_latex_formatted(frame: pd.DataFrame, caption: str, label: str, max_stds: dict) -> str:
    """Convert DataFrame to LaTeX table with best/second formatting."""
    lines = []
    lines.append("\\begin{table*}[htbp]")
    lines.append("\\centering")
    lines.append("\\small")
    lines.append("\\caption{" + caption + "}")
    lines.append("\\label{" + label + "}")
    lines.append("\\begin{tabular}{lccccc}")
    lines.append("\\toprule")
    lines.append("Method & Presence Acc. & IoU (BBox) & IoU@0.5 (BBox) & IoU (Mask) & IoU@0.5 (Mask) \\\\")
    lines.append("\\midrule")
    
    for method, row in frame.iterrows():
        cells = [method]
        for col in frame.columns:
            value = str(row[col])
            cells.append(value)
        lines.append(" & ".join(cells) + " \\\\")
    
    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("\\end{table*}")
    
    return "\n".join(lines)

# Create caption with max std information and formatting note
caption_text = (f"Bounding box evaluation results on CholecSeg8k (Zero-shot Combined) comparing both IoU metrics. "
                f"IoU (BBox) measures bbox-to-bbox overlap, IoU (Mask) measures bbox-to-segmentation overlap. "
                f"Best values are in \\textbf{{bold}}, second-best in \\textit{{italics}}. "
                f"Bootstrap standard deviations (1000 samples) were at most "
                f"{max_stds_simple['presence']:.3f} for Presence, "
                f"{max_stds_simple['iou_bbox']:.3f} for IoU (BBox), "
                f"{max_stds_simple['iou_at_50_bbox']:.3f} for IoU@0.5 (BBox), "
                f"{max_stds_simple['iou_mask']:.3f} for IoU (Mask), and "
                f"{max_stds_simple['iou_at_50_mask']:.3f} for IoU@0.5 (Mask).")

latex_table = dataframe_to_latex_formatted(
    df_formatted,  # Use the formatted dataframe
    caption_text,
    "tab:bbox_results_formatted",
    max_stds_simple
)

print("\n" + "="*80)
print("LATEX TABLE (With Best/Second-Best Formatting)")
print("="*80)
print(latex_table)

# Save to latex folder
import os
latex_dir = "latex"
os.makedirs(latex_dir, exist_ok=True)
latex_file = os.path.join(latex_dir, "bbox_results_table_formatted.tex")

with open(latex_file, "w") as f:
    f.write(latex_table)
print(f"\n✅ Formatted LaTeX table saved to {latex_file}")

# Show which models are best/second for each metric
print("\n" + "="*80)
print("BEST AND SECOND-BEST MODELS PER METRIC")
print("="*80)
for col in best_second:
    if best_second[col]['best'] is not None:
        best_val = best_second[col]['best']
        second_val = best_second[col]['second']
        
        # Find models with these values
        best_models = df_raw[df_raw[col] == best_val].index.tolist()
        second_models = df_raw[df_raw[col] == second_val].index.tolist() if second_val else []
        
        print(f"\n{col[1]}:")
        print(f"  Best ({best_val:.3f}): {', '.join(best_models)}")
        if second_models:
            print(f"  Second ({second_val:.3f}): {', '.join(second_models)}")


LATEX TABLE (With Best/Second-Best Formatting)
\begin{table*}[htbp]
\centering
\small
\caption{Bounding box evaluation results on CholecSeg8k (Zero-shot Combined) comparing both IoU metrics. IoU (BBox) measures bbox-to-bbox overlap, IoU (Mask) measures bbox-to-segmentation overlap. Best values are in \textbf{bold}, second-best in \textit{italics}. Bootstrap standard deviations (1000 samples) were at most 0.010 for Presence, 0.009 for IoU (BBox), 0.015 for IoU@0.5 (BBox), 0.007 for IoU (Mask), and 0.011 for IoU@0.5 (Mask).}
\label{tab:bbox_results_formatted}
\begin{tabular}{lccccc}
\toprule
Method & Presence Acc. & IoU (BBox) & IoU@0.5 (BBox) & IoU (Mask) & IoU@0.5 (Mask) \\
\midrule
GPT-4.1 & \textbf{0.744} & \textit{0.341} & \textit{0.256} & \textbf{0.274} & \textit{0.106} \\
Gemini-2.0-Flash & 0.655 & 0.294 & 0.192 & 0.208 & 0.075 \\
Claude-Sonnet-4 & 0.688 & 0.267 & 0.122 & 0.179 & 0.048 \\
Llava-v1.6-Mistral-7B & 0.546 & 0.000 & 0.000 & 0.000 & 0.000 \\
Qwen2.5-VL-7B & 0.714 & \te

In [6]:
# -----------------------------
# 5) Create a prettier version with formatting (no percentages)
# -----------------------------

# Create version with bold for best results (raw numbers, no percentages)
df_pretty = df.copy()

# Convert to numeric for comparison (handle mean ± std format)
for col in df_pretty.columns:
    numeric_values = []
    for val in df_pretty[col]:
        if val != "—":
            # Extract just the mean value (before ±)
            mean_part = val.split(" ±")[0] if " ±" in val else val
            numeric_values.append(float(mean_part))
    
    if numeric_values:
        best_val = max(numeric_values)
        
        # Bold the best value
        for idx in df_pretty.index:
            val = df_pretty.loc[idx, col]
            if val != "—":
                # Extract mean value
                mean_part = val.split(" ±")[0] if " ±" in val else val
                num_val = float(mean_part)
                
                # Bold if best (compare only mean values)
                if num_val == best_val:
                    df_pretty.loc[idx, col] = f"\\textbf{{{val}}}"

print("\n" + "="*60)
print("FORMATTED RESULTS TABLE (Raw Numbers, No Percentages)")
print("="*60)
print(df_pretty.to_string())


FORMATTED RESULTS TABLE (Raw Numbers, No Percentages)
Task                      CholecSeg8k                                                                
Metric                       Presence      IoU (BBox)  IoU@0.5 (BBox)      IoU (Mask)  IoU@0.5 (Mask)
GPT-4.1                \textbf{0.744}           0.341           0.256  \textbf{0.274}           0.106
Gemini-2.0-Flash                0.655           0.294           0.192           0.208           0.075
Claude-Sonnet-4                 0.688           0.267           0.122           0.179           0.048
Llava-v1.6-Mistral-7B           0.546           0.000           0.000           0.000           0.000
Qwen2.5-VL-7B                   0.714  \textbf{0.367}  \textbf{0.342}           0.216  \textbf{0.118}
Pixtral-12B                     0.715           0.090           0.009           0.078           0.000


In [7]:
# -----------------------------
# 6) Compare different evaluation modes with both IoU types
# -----------------------------

modes = ["zeroshot_combined", "zeroshot_separate", "fewshot_combined", "fewshot_separate"]
all_mode_results = {}

print("Loading all modes with BOTH IoU types...")
for mode in modes:
    print(f"\nLoading {mode}...")
    mode_results = load_summary_results(mode, compute_bootstrap=False)  # Skip bootstrap for comparison
    if mode_results:
        all_mode_results[mode] = mode_results

# Create comparison table
if all_mode_results:
    print("\n" + "="*100)
    print("COMPARISON ACROSS EVALUATION MODES (BOTH IoU TYPES)")
    print("="*100)
    
    for model_name in ALL_METHODS:
        print(f"\n{model_name}:")
        for mode, results in all_mode_results.items():
            if model_name in results:
                metrics = results[model_name]
                presence = metrics.get('presence_accuracy', 0)
                iou_bbox = metrics.get('mean_iou_bbox', 0)
                iou_mask = metrics.get('mean_iou_mask', 0)
                iou50_bbox = metrics.get('iou_at_50_bbox', 0)
                iou50_mask = metrics.get('iou_at_50_mask', 0)
                print(f"  {mode:20s}: Pres={presence:.3f}, IoU-B={iou_bbox:.3f}, IoU-M={iou_mask:.3f}, IoU@0.5-B={iou50_bbox:.3f}, IoU@0.5-M={iou50_mask:.3f}")
            else:
                print(f"  {mode:20s}: No results")

Loading all modes with BOTH IoU types...

Loading zeroshot_combined...
Loaded 200 files for GPT-4.1 (both IoU types)
Loaded 200 files for Gemini-2.0-Flash (both IoU types)
Loaded 200 files for Llava-v1.6-Mistral-7B (both IoU types)
Loaded 200 files for Claude-Sonnet-4 (both IoU types)
Loaded 200 files for Pixtral-12B (both IoU types)
Loaded 200 files for Qwen2.5-VL-7B (both IoU types)

Loading zeroshot_separate...
Loaded 200 files for Claude-Sonnet-4 (both IoU types)
Loaded 200 files for GPT-4.1 (both IoU types)
Loaded 200 files for Gemini-2.0-Flash (both IoU types)
Loaded 1 files for Llava-v1.6-Mistral-7B (both IoU types)

Loading fewshot_combined...
Loaded 200 files for Claude-Sonnet-4 (both IoU types)
Loaded 200 files for Llava-v1.6-Mistral-7B (both IoU types)
Loaded 200 files for Qwen2.5-VL-7B (both IoU types)
Loaded 200 files for Gemini-2.0-Flash (both IoU types)
Loaded 200 files for Pixtral-12B (both IoU types)
Loaded 200 files for GPT-4.1 (both IoU types)

Loading fewshot_separa

In [8]:
# -----------------------------
# 7) Summary statistics
# -----------------------------

print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

if zeroshot_results:
    # Compute averages across models
    presence_scores = [r['presence_accuracy'] for r in zeroshot_results.values() if 'presence_accuracy' in r]
    iou_scores = [r['mean_iou'] for r in zeroshot_results.values() if 'mean_iou' in r]
    iou50_scores = [r['iou_at_50'] for r in zeroshot_results.values() if 'iou_at_50' in r]
    
    print("\nZero-shot Combined Results:")
    if presence_scores:
        print(f"  Presence Accuracy:")
        print(f"    Mean: {np.mean(presence_scores):.3f}")
        print(f"    Std:  {np.std(presence_scores):.3f}")
        print(f"    Min:  {np.min(presence_scores):.3f}")
        print(f"    Max:  {np.max(presence_scores):.3f}")
    
    if iou_scores:
        print(f"\n  Mean IoU:")
        print(f"    Mean: {np.mean(iou_scores):.3f}")
        print(f"    Std:  {np.std(iou_scores):.3f}")
        print(f"    Min:  {np.min(iou_scores):.3f}")
        print(f"    Max:  {np.max(iou_scores):.3f}")
    
    if iou50_scores:
        print(f"\n  IoU@0.5:")
        print(f"    Mean: {np.mean(iou50_scores):.3f}")
        print(f"    Std:  {np.std(iou50_scores):.3f}")
        print(f"    Min:  {np.min(iou50_scores):.3f}")
        print(f"    Max:  {np.max(iou50_scores):.3f}")
    
    # Find best model for each metric
    print("\n  Best Models:")
    if presence_scores:
        best_presence = max(zeroshot_results.items(), key=lambda x: x[1].get('presence_accuracy', 0))
        print(f"    Presence: {best_presence[0]} ({best_presence[1]['presence_accuracy']:.3f})")
    
    if iou_scores:
        best_iou = max(zeroshot_results.items(), key=lambda x: x[1].get('mean_iou', 0))
        print(f"    Mean IoU: {best_iou[0]} ({best_iou[1]['mean_iou']:.3f})")
    
    if iou50_scores:
        best_iou50 = max(zeroshot_results.items(), key=lambda x: x[1].get('iou_at_50', 0))
        print(f"    IoU@0.5:  {best_iou50[0]} ({best_iou50[1]['iou_at_50']:.3f})")


SUMMARY STATISTICS

Zero-shot Combined Results:
  Presence Accuracy:
    Mean: 0.677
    Std:  0.064
    Min:  0.546
    Max:  0.744

  Best Models:
    Presence: GPT-4.1 (0.744)


In [9]:
# -----------------------------
# 8) Comprehensive Main Table with Model Types and Both IoU Types
# -----------------------------

print("\n" + "="*60)
print("COMPREHENSIVE MAIN TABLE - With Model Categories")
print("="*60)

# Define model categories and their members
MODEL_CATEGORIES = {
    "Commercial LVLMs": ["GPT-4.1", "Gemini-2.0-Flash", "Claude-Sonnet-4"],
    "Open-Source LVLMs": ["Llava-v1.6-Mistral-7B", "Qwen2.5-VL-7B", "Pixtral-12B"],
    "CLIP-based Models": ["PeskaVLP", "RASO"],
    "Task-Specific Models": ["CholeNet", "GoNoGoNet"]
}

# Create ordered list of all models
ALL_METHODS_ORDERED = []
for category in MODEL_CATEGORIES:
    ALL_METHODS_ORDERED.extend(MODEL_CATEGORIES[category])

# Tasks with both IoU types for localization
TASKS_FULL = {
    "CholecOrgans": ["Presence", "IoU-B", "IoU-M"],
    "CholecGoNoGo": ["Presence", "IoU-B", "IoU-M"],
    "CholecSeg8k":  ["Presence", "IoU-B", "IoU-M"],
}

# Build the full table structure
import itertools
columns_full = pd.MultiIndex.from_tuples(
    list(itertools.chain.from_iterable(
        [[(task, metric) for metric in TASKS_FULL[task]] for task in TASKS_FULL]
    )),
    names=["Task", "Metric"]
)

# Initialize dataframe with empty strings
df_full = pd.DataFrame("", index=ALL_METHODS_ORDERED, columns=columns_full)
df_full_raw = pd.DataFrame(np.nan, index=ALL_METHODS_ORDERED, columns=columns_full)  # For numeric comparisons

# Track maximum standard deviations for reporting
max_stds = {
    'presence': 0.0,
    'iou_bbox': 0.0,
    'iou_mask': 0.0
}

# Mark N/A cells based on model capabilities
NA_STR = "—"

def is_applicable(method: str, task: str, metric: str) -> bool:
    # Commercial and Open-Source LVLMs can do all tasks
    if method in MODEL_CATEGORIES["Commercial LVLMs"] + MODEL_CATEGORIES["Open-Source LVLMs"]:
        return True
    # CLIP-based models can do presence but not IoU
    if method in MODEL_CATEGORIES["CLIP-based Models"]:
        return (metric == "Presence")
    # Task-specific models
    if method == "CholeNet":
        return (task == "CholecOrgans" and metric == "Presence")
    if method == "GoNoGoNet":
        return (task == "CholecGoNoGo" and metric == "Presence")
    return False

# Fill N/A cells
for method in df_full.index:
    for (task, metric) in df_full.columns:
        if not is_applicable(method, task, metric):
            df_full.loc[method, (task, metric)] = NA_STR
            df_full_raw.loc[method, (task, metric)] = np.nan

# Fill in the actual data we have (only CholecSeg8k, zero-shot combined)
for model_name in ALL_METHODS_ORDERED:
    if model_name in zeroshot_results:
        metrics = zeroshot_results[model_name]
        
        # CholecSeg8k metrics - store only mean values, track max std
        if 'presence_accuracy' in metrics:
            mean_val = metrics['presence_accuracy']
            df_full.loc[model_name, ("CholecSeg8k", "Presence")] = f"{mean_val:.3f}"
            df_full_raw.loc[model_name, ("CholecSeg8k", "Presence")] = mean_val
            if 'presence_accuracy_std' in metrics:
                max_stds['presence'] = max(max_stds['presence'], metrics['presence_accuracy_std'])
        
        # Bbox-to-bbox IoU
        if 'mean_iou_bbox' in metrics:
            mean_val = metrics['mean_iou_bbox']
            df_full.loc[model_name, ("CholecSeg8k", "IoU-B")] = f"{mean_val:.3f}"
            df_full_raw.loc[model_name, ("CholecSeg8k", "IoU-B")] = mean_val
            if 'mean_iou_bbox_std' in metrics:
                max_stds['iou_bbox'] = max(max_stds['iou_bbox'], metrics['mean_iou_bbox_std'])
        
        # Bbox-to-mask IoU
        if 'mean_iou_mask' in metrics:
            mean_val = metrics['mean_iou_mask']
            df_full.loc[model_name, ("CholecSeg8k", "IoU-M")] = f"{mean_val:.3f}"
            df_full_raw.loc[model_name, ("CholecSeg8k", "IoU-M")] = mean_val
            if 'mean_iou_mask_std' in metrics:
                max_stds['iou_mask'] = max(max_stds['iou_mask'], metrics['mean_iou_mask_std'])

# Find best and second-best for each column
best_second_best = {}
for col in df_full_raw.columns:
    col_values = df_full_raw[col].dropna()
    if len(col_values) >= 2:
        sorted_values = col_values.nlargest(2)
        best_second_best[col] = {
            'best': sorted_values.iloc[0],
            'second': sorted_values.iloc[1]
        }
    elif len(col_values) == 1:
        best_second_best[col] = {
            'best': col_values.iloc[0],
            'second': None
        }

# Apply formatting to best and second-best values
df_formatted = df_full.copy()
for col in df_full_raw.columns:
    if col in best_second_best:
        for idx in df_full_raw.index:
            if not pd.isna(df_full_raw.loc[idx, col]):
                value = df_full_raw.loc[idx, col]
                formatted_value = df_full.loc[idx, col]
                
                if value == best_second_best[col]['best']:
                    # Bold the best value
                    df_formatted.loc[idx, col] = f"\\textbf{{{formatted_value}}}"
                elif best_second_best[col]['second'] is not None and value == best_second_best[col]['second']:
                    # Italicize the second-best value
                    df_formatted.loc[idx, col] = f"\\textit{{{formatted_value}}}"

print("\nData-filled table (only CholecSeg8k has actual values):")
print(df_full.to_string())

print("\nMaximum bootstrap standard deviations:")
print(f"  Presence: {max_stds['presence']:.3f}")
print(f"  IoU-B: {max_stds['iou_bbox']:.3f}")
print(f"  IoU-M: {max_stds['iou_mask']:.3f}")

# Generate LaTeX table with category rows
def generate_latex_with_category_rows(df, model_categories, max_stds):
    lines = []
    
    # Table header
    lines.append("\\begin{table*}[t]")
    lines.append("\\centering")
    lines.append("\\small")
    lines.append("\\setlength{\\tabcolsep}{4pt}")
    
    # Calculate number of columns
    n_data_cols = len(df.columns)
    col_spec = "l" + "c" * n_data_cols  # Left-aligned for method, centered for data
    
    lines.append(f"\\begin{{tabular}}{{{col_spec}}}")
    lines.append("\\toprule")
    
    # First header row: dataset names
    first_row = ["\\multirow{2}{*}{Method}"]
    for task in TASKS_FULL:
        n = len(TASKS_FULL[task])
        first_row.append(f"\\multicolumn{{{n}}}{{c}}{{\\textbf{{{task}}}}}")
    lines.append(" & ".join(first_row) + " \\\\")
    
    # Second header row: metrics
    second_row = [""]  # Empty for Method column
    for task in TASKS_FULL:
        for metric in TASKS_FULL[task]:
            if metric == "IoU-B":
                second_row.append("IoU-B$^a$")
            elif metric == "IoU-M":
                second_row.append("IoU-M$^b$")
            else:
                second_row.append(metric)
    lines.append(" & ".join(second_row) + " \\\\")
    lines.append("\\midrule")
    
    # Table body with category rows
    for cat_idx, (category, models) in enumerate(model_categories.items()):
        # Add category header row spanning all columns
        n_cols_total = 1 + n_data_cols  # Method column + data columns
        lines.append(f"\\multicolumn{{{n_cols_total}}}{{l}}{{\\textit{{{category}}}}} \\\\")
        
        # Add each model in the category
        for model in models:
            if model in df.index:
                row_cells = []
                
                # Add model name in bold
                row_cells.append(f"\\textbf{{{model}}}")
                
                # Add data values (with best/second formatting)
                for col in df.columns:
                    value = str(df.loc[model, col])
                    row_cells.append(value)
                
                lines.append(" & ".join(row_cells) + " \\\\")
        
        # Add spacing between categories (except after last)
        if cat_idx < len(model_categories) - 1:
            lines.append("\\addlinespace[3pt]")  # Add small vertical space
    
    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    
    # Caption with max standard deviations reported
    caption = ("Comparison of surgical organ detection models grouped by type. "
               "Commercial and Open-Source LVLMs can perform all tasks; "
               "CLIP-based models support presence detection only; "
               "Task-Specific models are limited to their designated datasets. "
               "Data shown is from zero-shot combined setting on CholecSeg8k. "
               "Best values are in \\textbf{bold}, second-best in \\textit{italics}. "
               f"Bootstrap standard deviations (1000 samples) were at most "
               f"{max_stds['presence']:.3f} for Presence, "
               f"{max_stds['iou_bbox']:.3f} for IoU-B, and "
               f"{max_stds['iou_mask']:.3f} for IoU-M. "
               "$^a$IoU-B: bbox-to-bbox IoU. $^b$IoU-M: bbox-to-mask IoU.")
    
    lines.append(f"\\caption{{{caption}}}")
    lines.append("\\label{tab:main_results}")
    lines.append("\\end{table*}")
    
    return "\n".join(lines)

# Generate the LaTeX code using formatted dataframe
latex_code = generate_latex_with_category_rows(df_formatted, MODEL_CATEGORIES, max_stds)

print("\n" + "="*80)
print("LATEX TABLE - WITH BEST/SECOND-BEST FORMATTING")
print("="*80)
print(latex_code)

# Save to latex folder
import os
latex_dir = "latex"
os.makedirs(latex_dir, exist_ok=True)
latex_file = os.path.join(latex_dir, "main_table_categorized_formatted.tex")

with open(latex_file, "w") as f:
    f.write(latex_code)
print(f"\n✅ Formatted LaTeX table saved to {latex_file}")

# Also create a simplified view for display
print("\n" + "="*80)
print("BEST AND SECOND-BEST PER COLUMN")
print("="*80)

for col in best_second_best:
    if col[0] == "CholecSeg8k":  # Only show for columns with data
        best_val = best_second_best[col]['best']
        second_val = best_second_best[col]['second']
        
        # Find which models have these values
        best_models = df_full_raw[df_full_raw[col] == best_val].index.tolist()
        second_models = df_full_raw[df_full_raw[col] == second_val].index.tolist() if second_val else []
        
        print(f"\n{col[0]} - {col[1]}:")
        print(f"  Best ({best_val:.3f}): {', '.join(best_models)}")
        if second_models:
            print(f"  Second ({second_val:.3f}): {', '.join(second_models)}")


COMPREHENSIVE MAIN TABLE - With Model Categories

Data-filled table (only CholecSeg8k has actual values):
Task                  CholecOrgans             CholecGoNoGo             CholecSeg8k              
Metric                    Presence IoU-B IoU-M     Presence IoU-B IoU-M    Presence  IoU-B  IoU-M
GPT-4.1                                                                       0.744  0.341  0.274
Gemini-2.0-Flash                                                              0.655  0.294  0.208
Claude-Sonnet-4                                                               0.688  0.267  0.179
Llava-v1.6-Mistral-7B                                                         0.546  0.000  0.000
Qwen2.5-VL-7B                                                                 0.714  0.367  0.216
Pixtral-12B                                                                   0.715  0.090  0.078
PeskaVLP                               —     —                  —     —                  —      —
RASO       

In [12]:
# -----------------------------
# 10) Ablation Study for API Models (Zero-shot/Few-shot × Combined/Separate)
# Compact one-column LaTeX output
# -----------------------------

print("\n" + "="*80)
print("ABLATION STUDY: API MODELS ACROSS ALL EVALUATION SETTINGS")
print("="*80)

# Define API models to analyze
api_models = ["GPT-4.1", "Claude-Sonnet-4", "Gemini-2.0-Flash"]

# Define evaluation modes
modes = ["zeroshot_combined", "zeroshot_separate", "fewshot_combined", "fewshot_separate"]
mode_display_names = {
    "zeroshot_combined": "Zero-shot Combined",
    "zeroshot_separate": "Zero-shot Separate", 
    "fewshot_combined": "Few-shot Combined",
    "fewshot_separate": "Few-shot Separate"
}

# --- Compact labels for LaTeX ---
short_mode_names = {
    "Zero-shot Combined": "ZS-Comb.",
    "Zero-shot Separate": "ZS-Sep.",
    "Few-shot Combined": "FS-Comb.",
    "Few-shot Separate": "FS-Sep."
}
col_header_map = {"Presence": "Pres.", "IoU (BBox)": "IoU-B", "IoU (Mask)": "IoU-M"}

# Columns for ablation table - focusing on key metrics
ablation_columns = ["Presence", "IoU (BBox)", "IoU (Mask)"]

# Create multi-level index for the ablation table (Model × Setting)
multi_index = []
for model in api_models:
    for mode in mode_display_names.values():
        multi_index.append((model, mode))
ablation_index = pd.MultiIndex.from_tuples(multi_index, names=["Model", "Setting"])

# Initialize dataframes
df_ablation_all = pd.DataFrame("", index=ablation_index, columns=ablation_columns)
df_ablation_raw = pd.DataFrame(np.nan, index=ablation_index, columns=ablation_columns)

# Track max stds across all models
ablation_max_stds = {col: 0.0 for col in ablation_columns}

# Load results for each API model
print("\nLoading results for API models...")
for model_name in api_models:
    print(f"\n{model_name}:")
    for mode in modes:
        display_mode = mode_display_names[mode]
        print(f"  Loading {display_mode}...")
        mode_results = load_summary_results(mode, compute_bootstrap=True)
        if model_name in mode_results:
            metrics = mode_results[model_name]
            # Presence accuracy
            if 'presence_accuracy' in metrics:
                value = metrics['presence_accuracy']
                df_ablation_all.loc[(model_name, display_mode), "Presence"] = f"{value:.3f}"
                df_ablation_raw.loc[(model_name, display_mode), "Presence"] = value
                if 'presence_accuracy_std' in metrics:
                    ablation_max_stds["Presence"] = max(ablation_max_stds["Presence"], metrics['presence_accuracy_std'])
            # Bbox-to-bbox IoU
            if 'mean_iou_bbox' in metrics:
                value = metrics['mean_iou_bbox']
                df_ablation_all.loc[(model_name, display_mode), "IoU (BBox)"] = f"{value:.3f}"
                df_ablation_raw.loc[(model_name, display_mode), "IoU (BBox)"] = value
                if 'mean_iou_bbox_std' in metrics:
                    ablation_max_stds["IoU (BBox)"] = max(ablation_max_stds["IoU (BBox)"], metrics['mean_iou_bbox_std'])
            # Bbox-to-mask IoU
            if 'mean_iou_mask' in metrics:
                value = metrics['mean_iou_mask']
                df_ablation_all.loc[(model_name, display_mode), "IoU (Mask)"] = f"{value:.3f}"
                df_ablation_raw.loc[(model_name, display_mode), "IoU (Mask)"] = value
                if 'mean_iou_mask_std' in metrics:
                    ablation_max_stds["IoU (Mask)"] = max(ablation_max_stds["IoU (Mask)"], metrics['mean_iou_mask_std'])
        else:
            print(f"    No results found")
            for col in ablation_columns:
                df_ablation_all.loc[(model_name, display_mode), col] = "—"

# Find best value for each column (across all models and settings)
ablation_best = {}
for col in df_ablation_raw.columns:
    col_values = df_ablation_raw[col].dropna()
    if len(col_values) > 0:
        ablation_best[col] = col_values.max()

# Apply formatting to best values (global best across all models)
df_ablation_formatted = df_ablation_all.copy()
for col in df_ablation_raw.columns:
    if col in ablation_best:
        for idx in df_ablation_raw.index:
            if not pd.isna(df_ablation_raw.loc[idx, col]):
                value = df_ablation_raw.loc[idx, col]
                formatted_value = df_ablation_all.loc[idx, col]
                if value == ablation_best[col]:
                    df_ablation_formatted.loc[idx, col] = f"\\textbf{{{formatted_value}}}"

print("\n" + "="*80)
print("ABLATION TABLE: ALL API MODELS (raw values)")
print("="*80)
print(df_ablation_all.to_string())

# Analyze improvements for each model
print("\n" + "="*80)
print("PERFORMANCE ANALYSIS")
print("="*80)
for model_name in api_models:
    print(f"\n{model_name}:")
    # Zero-shot → Few-shot (Combined)
    zs_comb = (model_name, "Zero-shot Combined")
    fs_comb = (model_name, "Few-shot Combined")
    if zs_comb in df_ablation_raw.index and fs_comb in df_ablation_raw.index:
        print("  Zero-shot → Few-shot (Combined):")
        for col in ablation_columns:
            zs_val = df_ablation_raw.loc[zs_comb, col]
            fs_val = df_ablation_raw.loc[fs_comb, col]
            if not pd.isna(zs_val) and not pd.isna(fs_val) and zs_val != 0:
                improvement = (fs_val - zs_val) / zs_val * 100
                print(f"    {col}: {zs_val:.3f} → {fs_val:.3f} ({improvement:+.1f}%)")
    # Combined → Separate (Zero-shot)
    zs_sep = (model_name, "Zero-shot Separate")
    if zs_comb in df_ablation_raw.index and zs_sep in df_ablation_raw.index:
        print("  Combined → Separate (Zero-shot):")
        for col in ablation_columns:
            comb_val = df_ablation_raw.loc[zs_comb, col]
            sep_val = df_ablation_raw.loc[zs_sep, col]
            if not pd.isna(comb_val) and not pd.isna(sep_val) and comb_val != 0:
                change = (sep_val - comb_val) / comb_val * 100
                print(f"    {col}: {comb_val:.3f} → {sep_val:.3f} ({change:+.1f}%)")

# -------- Compact, one-column LaTeX generator --------
def generate_api_ablation_latex_compact(df, max_stds, short_mode_names, col_header_map,
                                        decimals=2, tabcolsep_pt=3, fontsize='scriptsize'):
    """
    Render a compact, one-column LaTeX table with short headers/labels and tight spacing.
    Requires \\usepackage{booktabs} and \\usepackage{multirow} in your preamble.
    """
    # header with short column names
    header_cols = [col_header_map.get(c, c) for c in df.columns]

    lines = []
    lines.append("\\begin{table}[t]")     # one-column
    lines.append("\\centering")
    lines.append(f"\\{fontsize}")         # e.g., \\scriptsize / \\tiny
    lines.append(f"\\setlength{{\\tabcolsep}}{{{tabcolsep_pt}pt}}")  # tighter spacing
    lines.append("\\begin{tabular}{llccc}")
    lines.append("\\toprule")
    lines.append(f"Model & Setting & {header_cols[0]} & {header_cols[1]} & {header_cols[2]} \\\\")
    lines.append("\\midrule")

    current_model = None
    first_block = True

    def fmt_cell(x):
        """
        Keep \\textbf{...} if present, while controlling decimals.
        """
        s = str(x)
        # bold-wrapped number
        if s.startswith("\\textbf{") and s.endswith("}"):
            inner = s[len("\\textbf{"):-1]
            try:
                return f"\\textbf{{{float(inner):.{decimals}f}}}"
            except Exception:
                return s
        # plain number string
        try:
            return f"{float(s):.{decimals}f}"
        except Exception:
            return s  # for dashes, etc.

    for (model, setting), row in df.iterrows():
        # new model block
        if model != current_model:
            if not first_block:
                lines.append("\\midrule")
            first_block = False
            current_model = model
            model_cell = f"\\multirow{{4}}{{*}}{{\\textbf{{{model}}}}}"
            left = model_cell
        else:
            left = ""

        # compact setting label
        setting_short = short_mode_names.get(setting, setting)

        pres = fmt_cell(row[df.columns[0]])
        iou_b = fmt_cell(row[df.columns[1]])
        iou_m = fmt_cell(row[df.columns[2]])

        lines.append(f"{left} & {setting_short} & {pres} & {iou_b} & {iou_m} \\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")

    caption = (f"Ablation of API models on CholecSeg8k. Best values in \\textbf{{bold}}. "
               f"Max bootstrap stds (1000 samples): "
               f"{col_header_map['Presence']} {max_stds['Presence']:.3f}, "
               f"{col_header_map['IoU (BBox)']} {max_stds['IoU (BBox)']:.3f}, "
               f"{col_header_map['IoU (Mask)']} {max_stds['IoU (Mask)']:.3f}.")
    lines.append(f"\\caption{{{caption}}}")
    lines.append("\\label{{tab:ablation_api_models}}")
    lines.append("\\end{table}")

    return "\n".join(lines)

# Generate compact LaTeX
latex_ablation = generate_api_ablation_latex_compact(
    df_ablation_formatted, ablation_max_stds, short_mode_names, col_header_map,
    decimals=2, tabcolsep_pt=3, fontsize='small'
)

print("\n" + "="*80)
print("LATEX TABLE - ABLATION STUDY FOR API MODELS (COMPACT, ONE-COLUMN)")
print("="*80)
print(latex_ablation)

# Save to latex folder
latex_file = os.path.join(latex_dir, "ablation_api_models_compact.tex")
with open(latex_file, "w") as f:
    f.write(latex_ablation)
print(f"\n✅ Ablation study table saved to {latex_file}")

# Summary statistics across models
print("\n" + "="*80)
print("SUMMARY ACROSS API MODELS")
print("="*80)

# Average improvement from zero-shot to few-shot
improvements = []
for model in api_models:
    for col in ablation_columns:
        zs_val = df_ablation_raw.loc[(model, "Zero-shot Combined"), col]
        fs_val = df_ablation_raw.loc[(model, "Few-shot Combined"), col]
        if not pd.isna(zs_val) and not pd.isna(fs_val) and zs_val != 0:
            improvements.append((fs_val - zs_val) / zs_val * 100)

if improvements:
    print(f"\nAverage improvement from Zero-shot to Few-shot (Combined):")
    print(f"  Mean: {np.mean(improvements):+.1f}%")
    print(f"  Std:  {np.std(improvements):.1f}%")



ABLATION STUDY: API MODELS ACROSS ALL EVALUATION SETTINGS

Loading results for API models...

GPT-4.1:
  Loading Zero-shot Combined...
Loaded 200 files for GPT-4.1 (both IoU types)
Loaded 200 files for Gemini-2.0-Flash (both IoU types)
Loaded 200 files for Llava-v1.6-Mistral-7B (both IoU types)
Loaded 200 files for Claude-Sonnet-4 (both IoU types)
Loaded 200 files for Pixtral-12B (both IoU types)
Loaded 200 files for Qwen2.5-VL-7B (both IoU types)
  Loading Zero-shot Separate...
Loaded 200 files for Claude-Sonnet-4 (both IoU types)
Loaded 200 files for GPT-4.1 (both IoU types)
Loaded 200 files for Gemini-2.0-Flash (both IoU types)
Loaded 1 files for Llava-v1.6-Mistral-7B (both IoU types)
  Loading Few-shot Combined...
Loaded 200 files for Claude-Sonnet-4 (both IoU types)
Loaded 200 files for Llava-v1.6-Mistral-7B (both IoU types)
Loaded 200 files for Qwen2.5-VL-7B (both IoU types)
Loaded 200 files for Gemini-2.0-Flash (both IoU types)
Loaded 200 files for Pixtral-12B (both IoU types)
